<a href="https://colab.research.google.com/github/aashishneupanecs24-creator/BIOS-LAB-1BF24CS004/blob/main/Grey_WolfOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# ---------------------------------------------------------
# 1. Problem Definition: 3-Generator Economic Load Dispatch
# ---------------------------------------------------------
# Generator cost coefficients: F(P) = a*P^2 + b*P + c ($/hr)
a = np.array([0.001562, 0.00194, 0.00482])
b = np.array([7.92, 7.85, 7.97])
c = np.array([561.0, 310.0, 78.0])

# Generator active power limits (MW)
P_min = np.array([100.0, 100.0, 50.0])
P_max = np.array([600.0, 400.0, 200.0])

# Total system power demand (MW)
P_demand = 850.0


def objective_function(P):
    """Calculates total fuel cost plus a quadratic penalty for power mismatch."""
    # Fuel cost calculation
    fuel_cost = np.sum(a * (P**2) + b * P + c)

    # Power balance constraint: sum(P) == P_demand
    mismatch = abs(np.sum(P) - P_demand)
    penalty = 1000.0 * (mismatch**2)

    return fuel_cost + penalty


# ---------------------------------------------------------
# 2. Grey Wolf Optimizer (GWO) Implementation
# ---------------------------------------------------------
def grey_wolf_optimizer(obj_func, lb, ub, num_wolves=30, max_iter=300):
    dim = len(lb)

    # Initialize Alpha, Beta, and Delta wolves (top 3 solutions)
    alpha_pos = np.zeros(dim)
    alpha_score = float("inf")

    beta_pos = np.zeros(dim)
    beta_score = float("inf")

    delta_pos = np.zeros(dim)
    delta_score = float("inf")

    # Initialize wolf pack positions randomly within bounds
    positions = np.random.uniform(0, 1, (num_wolves, dim)) * (ub - lb) + lb
    convergence_curve = np.zeros(max_iter)

    # Main optimization loop
    for t in range(max_iter):
        for i in range(num_wolves):
            # Enforce lower and upper generator bounds
            positions[i] = np.clip(positions[i], lb, ub)

            # Evaluate fitness
            fitness = obj_func(positions[i])

            # Update Alpha, Beta, and Delta hierarchy
            if fitness < alpha_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = alpha_score, alpha_pos.copy()
                alpha_score, alpha_pos = fitness, positions[i].copy()
            elif fitness < beta_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = fitness, positions[i].copy()
            elif fitness < delta_score:
                delta_score, delta_pos = fitness, positions[i].copy()

        # Linearly decrease parameter 'a' from 2 to 0 (controls exploration vs exploitation)
        a_param = 2.0 - t * (2.0 / max_iter)

        # Update position of each wolf in the pack
        for i in range(num_wolves):
            for j in range(dim):
                # Encircling & hunting using Alpha
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2.0 * a_param * r1 - a_param
                C1 = 2.0 * r2
                D_alpha = abs(C1 * alpha_pos[j] - positions[i, j])
                X1 = alpha_pos[j] - A1 * D_alpha

                # Encircling & hunting using Beta
                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2.0 * a_param * r1 - a_param
                C2 = 2.0 * r2
                D_beta = abs(C2 * beta_pos[j] - positions[i, j])
                X2 = beta_pos[j] - A2 * D_beta

                # Encircling & hunting using Delta
                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2.0 * a_param * r1 - a_param
                C3 = 2.0 * r2
                D_delta = abs(C3 * delta_pos[j] - positions[i, j])
                X3 = delta_pos[j] - A3 * D_delta

                # Position update is the average guidance of the top 3 wolves
                positions[i, j] = (X1 + X2 + X3) / 3.0

        convergence_curve[t] = alpha_score

    return alpha_pos, alpha_score, convergence_curve


# ---------------------------------------------------------
# 3. Run Optimization
# ---------------------------------------------------------
if __name__ == "__main__":
    np.random.seed(42)
    best_dispatch, best_cost, _ = grey_wolf_optimizer(
        obj_func=objective_function,
        lb=P_min,
        ub=P_max,
        num_wolves=40,
        max_iter=500,
    )

    actual_fuel_cost = np.sum(a * (best_dispatch**2) + b * best_dispatch + c)
    print(f"Target Demand:      {P_demand:.2f} MW")
    print(f"Optimal Dispatch:   {np.round(best_dispatch, 2)} MW")
    print(f"Total Generated:    {np.sum(best_dispatch):.2f} MW")
    print(f"Minimum Fuel Cost:  ${actual_fuel_cost:.2f} / hr")

Target Demand:      850.00 MW
Optimal Dispatch:   [397.34 337.67 114.98] MW
Total Generated:    850.00 MW
Minimum Fuel Cost:  $8194.62 / hr
